# Security Log Analysis Pipeline

In [ ]:
# Install all pipeline dependencies
!pip install -q \
    pandas pyarrow \
    scikit-learn umap-learn \
    plotly \
    bertopic sentence-transformers hdbscan \
    chromadb \
    dspy-ai \
    nltk pyyaml requests tqdm \
    torch \
    ragas datasets \
    matplotlib seaborn \
    langchain-community langchain-ollama langchain-huggingface

import nltk
nltk.download('stopwords', quiet=True)


### Fetch and Extract Dataset

In [ ]:
import os
import zipfile
import glob

DATA_REPO_URL = "https://raw.githubusercontent.com/OTRF/Security-Datasets/master/datasets/compound/apt29/day1/apt29_evals_day1_manual.zip"
ZIP_PATH = "/content/apt29_evals_day1_manual.zip"
EXTRACT_DIR = "/content/data_path"

os.makedirs(EXTRACT_DIR, exist_ok=True)
!wget -q {DATA_REPO_URL} -O {ZIP_PATH}

print("extracting dataset")
with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
    zip_ref.extractall(EXTRACT_DIR)

DATA_PATH = glob.glob(f"{EXTRACT_DIR}/**/*.json", recursive=True)[0]
print(f"dataset loaded")


### Global configuration

In [ ]:
NORMALIZED_PARQUET = "/content/data/normalized.parquet"
LABELED_PARQUET    = "/content/data/labeled.parquet"
ANOMALIES_PARQUET  = "/content/data/anomalies.parquet"
TOPICS_PARQUET     = "/content/data/anomalies_with_topics.parquet"
CHROMA_DIR         = "/content/data/chroma_db"
RESULTS_JSON       = "/content/data/llm_results.json"
TOPIC_MODEL_DIR    = "/content/data/bertopic_model"
METRICS_JSON       = "/content/data/metrics_report.json"
LLM_METRICS_JSON   = "/content/data/llm_metrics.json"
FIGURES_DIR        = "/content/data/figures"

OLLAMA_MODEL       = "llama3"       # change model if needed
CONTAMINATION      = 0.05           # fraction flagged as anomalous
TOP_N_TOPICS       = 12             # topics passed to LLM
EVENTS_PER_TOPIC   = 3              # worst anomalies per topic

import os
os.makedirs("/content/data", exist_ok=True)
os.makedirs(FIGURES_DIR,    exist_ok=True)


---
## Stage 1 — Parse & Normalize


In [ ]:
"""
Stage 1: Parse & Normalize
Streams the NDJSON log file in chunks and produces a normalized Parquet file.
"""
import json, os, re
import pandas as pd
import numpy as np
from datetime import datetime
from tqdm.auto import tqdm


# ── Config ────────────────────────────────────────────────────────────────────
DATA_PATH         = "/content/data_path/apt29_evals_day1_manual_2020-05-01225525.json"
NORMALIZED_PARQUET = "/content/data/normalized.parquet"
CHUNK_SIZE        = 50_000          # rows kept in memory before flushing
# ──────────────────────────────────────────────────────────────────────────────


def _safe_int(val, default=0):
    try:
        return int(val)
    except (TypeError, ValueError):
        return default


def _hex_to_int(val):
    try:
        return int(val, 16) if isinstance(val, str) and val.lower().startswith("0x") else 0
    except ValueError:
        return 0


def _parse_dt(s):
    try:
        return datetime.strptime(s, "%Y-%m-%d %H:%M:%S")
    except Exception:
        return None


def _process_depth(path: str) -> int:
    return path.count("\\") if path else 0


def _basename(path: str) -> str:
    return path.split("\\")[-1].lower() if path else ""


def normalize_event(ev: dict) -> dict:
    """Flatten one raw Sysmon JSON event into a ML-ready dict."""
    dt   = _parse_dt(ev.get("EventTime", ""))
    img  = ev.get("Image") or ev.get("SourceImage") or ""
    timg = ev.get("TargetImage") or ""
    tobj = ev.get("TargetObject") or ""
    tfn  = ev.get("TargetFilename") or ""
    msg  = ev.get("Message") or ""

    return {
        # ── identifiers ──────────────────────────────────────────────────────
        "timestamp"        : dt.isoformat() if dt else "",
        "hour_of_day"      : dt.hour        if dt else -1,
        "day_of_week"      : dt.weekday()   if dt else -1,
        # ── event metadata ───────────────────────────────────────────────────
        "event_id"         : _safe_int(ev.get("EventID")),
        "channel"          : ev.get("Channel", ""),
        "source_name"      : ev.get("SourceName", ""),
        "severity"         : ev.get("Severity", ""),
        "severity_value"   : _safe_int(ev.get("SeverityValue")),
        "hostname"         : ev.get("Hostname", ""),
        # ── identity ─────────────────────────────────────────────────────────
        "account_name"     : ev.get("AccountName", ""),
        "domain"           : ev.get("Domain", ""),
        "is_system"        : 1 if ev.get("AccountName", "").upper() == "SYSTEM" else 0,
        # ── process / image ──────────────────────────────────────────────────
        "image"            : img,
        "image_base"       : _basename(img),
        "process_depth"    : _process_depth(img),
        "process_id"       : _safe_int(ev.get("ProcessId") or ev.get("SourceProcessId")),
        "target_image"     : timg,
        "target_image_base": _basename(timg),
        # ── access / registry / file ─────────────────────────────────────────
        "granted_access"   : _hex_to_int(ev.get("GrantedAccess", "0x0")),
        "target_object"    : tobj,
        "target_filename"  : tfn,
        # ── text ─────────────────────────────────────────────────────────────
        "message"          : msg,
        "message_len"      : len(msg),
    }


def parse_stage(data_path: str = DATA_PATH,
                out_path: str  = NORMALIZED_PARQUET,
                chunk_size: int = CHUNK_SIZE) -> pd.DataFrame:
    """
    Streams the NDJSON file line-by-line, normalizes each event,
    and saves a single consolidated Parquet file.
    Returns the final DataFrame.
    """
    os.makedirs(os.path.dirname(out_path), exist_ok=True)

    chunks, records, errors, total = [], [], 0, 0

    print(f"Streaming: {data_path}")
    with open(data_path, "r", encoding="utf-8", errors="replace") as fh:
        for line in tqdm(fh, desc="Parsing events", unit=" lines"):
            line = line.strip()
            if not line:
                continue
            try:
                ev = json.loads(line)
                records.append(normalize_event(ev))
                total += 1
            except json.JSONDecodeError:
                errors += 1
                continue

            if len(records) >= chunk_size:
                chunks.append(pd.DataFrame(records))
                records = []

    if records:
        chunks.append(pd.DataFrame(records))

    print(f"\nParsed {total:,} events | JSON errors: {errors:,}")

    df = pd.concat(chunks, ignore_index=True)

    # ── post-parse feature engineering ───────────────────────────────────────
    # One-hot top-15 EventIDs
    top_eids = df["event_id"].value_counts().head(15).index.tolist()
    for eid in top_eids:
        df[f"eid_{eid}"] = (df["event_id"] == eid).astype(np.int8)

    # Channel bucket
    df["channel_bucket"] = df["channel"].str.extract(r"(Sysmon|Security|System|Application)",
                                                       expand=False).fillna("Other")

    df.to_parquet(out_path, index=False)
    print(f"Saved -> {out_path}  ({df.shape[0]:,} rows x {df.shape[1]} cols)")
    return df

In [ ]:
df_norm = parse_stage(
    data_path  = DATA_PATH,
    out_path   = NORMALIZED_PARQUET,
    chunk_size = 50_000,
)
df_norm.head(3)


---
## Stage 0 — Ground Truth Labeling

Assigns a `ground_truth` label to every event in `normalized.parquet` using
a **three-tier rule set** derived from the MITRE ATT&CK APT29 technique mapping:

| Tier | Rule | Label |
|---|---|---|
| 1 — High confidence | EventID 10 targeting `lsass.exe`, EventID 13 writing to `\Run\` keys, etc. | `1` (Malicious) |
| 2 — Heuristic | Known offensive tool image names (`mimikatz.exe`, `psexec.exe`, etc.) | `1` (Malicious) |
| 3 — Default | All other events | `0` (Benign) |

> The OTRF APT29 Mordor dataset is **semi-labeled by design** — both adversarial and
> normal endpoint events are captured in the same file. No second dataset is needed.


In [ ]:
"""
Stage 0: Ground Truth Labeling
Reads normalized.parquet (produced by Stage 1) and adds a `ground_truth`
column using a three-tier rule set derived from the MITRE ATT&CK APT29
technique mapping and Sysmon event semantics.

Ground truth construction rationale:
  The OTRF APT29 Mordor dataset is semi-labeled by design.
  Each capture file contains both adversarial tradecraft events AND
  normal background endpoint activity generated during the same timeframe.
  Since no per-row binary label exists, we derive labels from:

    Tier 1 — High-confidence MITRE-mapped rules (EventID + target/process)
    Tier 2 — Known-bad process image names (heuristic)
    Tier 3 — Default  →  benign (0)

References:
  OTRF Threat Hunter Playbook: https://threathunterplaybook.com/
  MITRE ATT&CK APT29 group:    https://attack.mitre.org/groups/G0016/
"""
import os
import pandas as pd

NORMALIZED_PARQUET = "/content/data/normalized.parquet"
LABELED_PARQUET    = "/content/data/labeled.parquet"

# ── Tier 1 ────────────────────────────────────────────────────────────────────
# High-confidence rules: (EventID, field_name, substring_to_match)
# Derived from documented APT29 TTPs in the OTRF dataset and ATT&CK G0016.

TIER1_RULES = [
    # T1003.001 — LSASS Memory Credential Dumping
    (10,  "target_image",  "lsass.exe"),
    # T1547.001 — Registry Run Key Persistence
    (13,  "target_object", "\\CurrentVersion\\Run"),
    (13,  "target_object", "\\CurrentVersion\\RunOnce"),
    # T1059.001 — PowerShell execution chain (office apps spawning shells)
    (1,   "image_base",    "powershell.exe"),
    (1,   "image_base",    "cmd.exe"),
    # T1021.002 — SMB/Windows Admin Shares lateral movement
    (3,   "image_base",    "net.exe"),
    (3,   "image_base",    "net1.exe"),
    # T1055 — Process Injection
    (8,   "target_image",  ""),          # CreateRemoteThread — any target
    # T1543.003 — Windows Service creation
    (13,  "target_object", "\\Services\\"),
    # T1070.001 — Clear Windows Event Logs
    (1,   "image_base",    "wevtutil.exe"),
    # T1053.005 — Scheduled Task creation
    (1,   "image_base",    "schtasks.exe"),
    (11,  "target_filename","\\Tasks\\"),
    # T1027 — Obfuscated file write
    (11,  "target_filename", ".ps1"),
    (11,  "target_filename", ".vbs"),
    (11,  "target_filename", ".hta"),
    # T1218.011 — Signed Binary Proxy (rundll32/regsvr32)
    (1,   "image_base",    "rundll32.exe"),
    (1,   "image_base",    "regsvr32.exe"),
    # T1071.001 — Web Protocol C2
    (3,   "image_base",    "powershell.exe"),
    # T1140 — Deobfuscate via certutil
    (1,   "image_base",    "certutil.exe"),
    # T1036 — Masquerading (processes outside System32)
    (1,   "image_base",    "svchost.exe"),   # confirmed after path check below
]

# ── Tier 2 ────────────────────────────────────────────────────────────────────
# Known offensive tooling image basenames (heuristic).
KNOWN_BAD_IMAGES = {
    "mimikatz.exe", "psexec.exe", "psexecsvc.exe", "cobalt strike",
    "meterpreter", "empire", "covenant", "pwdump", "fgdump",
    "wce.exe", "gsecdump.exe", "lsadump", "procdump.exe",
    "rubeus.exe", "kerbrute.exe", "bloodhound.exe", "sharphound.exe",
    "invoke-mimikatz", "invoke-bloodhound",
}


def _apply_tier1(df: pd.DataFrame) -> pd.Series:
    """
    Vectorised application of Tier-1 rules.
    Returns a boolean Series: True = rule matched.
    """
    mask = pd.Series(False, index=df.index)

    for eid, field, substring in TIER1_RULES:
        eid_match = df["event_id"] == eid
        if not substring:
            # Match purely on EventID (e.g., EventID 8 = CreateRemoteThread)
            mask |= eid_match
        else:
            col = df[field].fillna("").str.lower()
            mask |= eid_match & col.str.contains(substring.lower(), regex=False)

    # Special case: EventID 1 svchost.exe NOT in System32 is suspicious
    # (legitimate svchost lives in C:\Windows\System32\svchost.exe)
    svchost_mask = (
        (df["event_id"] == 1)
        & df["image_base"].str.lower().eq("svchost.exe")
        & ~df["image"].str.lower().str.contains("system32", na=False)
    )
    mask |= svchost_mask

    return mask


def _apply_tier2(df: pd.DataFrame) -> pd.Series:
    """
    Returns a boolean Series: True = image_base is in the known-bad list.
    """
    return df["image_base"].str.lower().isin(KNOWN_BAD_IMAGES)


def label_stage(normalized_path: str = NORMALIZED_PARQUET,
                labeled_path: str   = LABELED_PARQUET) -> pd.DataFrame:
    """
    End-to-end Stage 0 entry point.
    Reads normalized.parquet, assigns ground_truth ∈ {0, 1}, saves labeled.parquet.
    Returns the labeled DataFrame.
    """
    os.makedirs(os.path.dirname(labeled_path), exist_ok=True)

    print("Loading normalized parquet...")
    df = pd.read_parquet(normalized_path)
    print(f"    Shape: {df.shape}")

    # ── Apply rules ───────────────────────────────────────────────────────────
    tier1_mask = _apply_tier1(df)
    tier2_mask = _apply_tier2(df)

    df["ground_truth"] = (tier1_mask | tier2_mask).astype(int)

    # ── Audit / diagnostics ───────────────────────────────────────────────────
    n_malicious = df["ground_truth"].sum()
    n_benign    = len(df) - n_malicious
    prevalence  = n_malicious / len(df) * 100

    print(f"\n  ── Ground Truth Distribution ───────────────────────────")
    print(f"     Benign    (0) : {n_benign:>8,}  ({100 - prevalence:.1f}%)")
    print(f"     Malicious (1) : {n_malicious:>8,}  ({prevalence:.1f}%)")
    print(f"     Total         : {len(df):>8,}")

    # Breakdown by which EventIDs were flagged
    malicious_df = df[df["ground_truth"] == 1]
    eid_breakdown = (
        malicious_df.groupby("event_id")
        .agg(count=("event_id", "size"), image_examples=("image_base", lambda x: list(x.unique())[:3]))
        .sort_values("count", ascending=False)
        .head(12)
    )
    print("\n  Top malicious EventIDs flagged:")
    print(eid_breakdown.to_string())
    print("  ────────────────────────────────────────────────────────\n")

    df.to_parquet(labeled_path, index=False)
    print(f"  Labeled dataset saved → {labeled_path}")

    return df

In [ ]:
df_labeled = label_stage(
    normalized_path = NORMALIZED_PARQUET,
    labeled_path    = LABELED_PARQUET,
)


In [ ]:
import pandas as pd
import plotly.express as px

df_labeled = pd.read_parquet(LABELED_PARQUET)

# Class distribution bar chart
counts = df_labeled["ground_truth"].value_counts().rename({0: "Benign", 1: "Malicious"})
fig = px.bar(
    x=counts.index,
    y=counts.values,
    color=counts.index,
    color_discrete_map={"Benign": "#00b4d8", "Malicious": "#e94560"},
    title="Ground Truth Class Distribution (APT29 Mordor Dataset)",
    labels={"x": "Class", "y": "Event Count"},
    template="plotly_dark",
    height=380,
)
fig.show()
print(f"Total events   : {len(df_labeled):,}")
print(f"Benign (0)     : {(df_labeled['ground_truth']==0).sum():,}")
print(f"Malicious (1)  : {(df_labeled['ground_truth']==1).sum():,}")


In [ ]:
import pandas as pd
import plotly.express as px

df_norm = pd.read_parquet(NORMALIZED_PARQUET)

fig = px.histogram(
    df_norm, x="event_id",
    color="is_system",
    title="Event ID Distribution (System vs User Accounts)",
    template="plotly_dark",
    barmode="overlay",
    opacity=0.8,
    height=420,
)
fig.show()

print(f"Total events : {len(df_norm):,}")
print(f"Unique hosts : {df_norm['hostname'].nunique()}")
print(f"Event ID types: {df_norm['event_id'].nunique()}")
print(df_norm[['event_id','hostname','channel','severity','message_len']].describe())


---
## Stage 2 — Comparative Anomaly Detection

Runs **three independent unsupervised anomaly detectors** on the same feature matrix:

| Model | Algorithm | Key Property |
|---|---|---|
| A | **Isolation Forest** | Ensemble tree-based; efficient on high-dimensional data |
| B | **One-Class SVM** | Kernel-based; non-linear decision boundary |
| C | **Deep Autoencoder** | Reconstruction-error; learns compact latent representation (GPU) |

Each model produces its own anomaly score column. The **union** of all three flags
forms the final anomaly set passed to downstream stages.

> Ground truth labels (from Stage 0) are carried through but **not used** for training —
> all models remain fully unsupervised. Labels are used only in Stage 5 evaluation.


In [ ]:
"""
Stage 2: Anomaly Detection — Comparative Multi-Model
Loads labeled.parquet, scores every event with three detectors:

  Model A — Isolation Forest  (existing, ensemble tree-based)
  Model B — One-Class SVM     (kernel-based, supports non-linear boundaries)
  Model C — Deep Autoencoder  (reconstruction-error, PyTorch / CUDA)

Each model adds its own anomaly score column. The union of events flagged
by ANY model is saved to anomalies.parquet.

A UMAP 2-D projection is rendered coloured by Isolation Forest score.

Research rationale:
  A thesis requires comparative evaluation rather than a single algorithm.
  Running all three on the same feature matrix allows a direct comparison
  of Precision / Recall / F1 / ROC-AUC in Stage 5 (stage5_evaluation.py).
"""
import os
import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM
from sklearn.preprocessing import StandardScaler
import umap

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

LABELED_PARQUET    = "/content/data/labeled.parquet"
ANOMALIES_PARQUET  = "/content/data/anomalies.parquet"
CONTAMINATION      = 0.05
UMAP_SAMPLE        = 60_000
RANDOM_STATE       = 42

# ── Autoencoder hyperparameters ───────────────────────────────────────────────
AE_EPOCHS          = 30
AE_BATCH_SIZE      = 1024
AE_LR              = 1e-3
AE_HIDDEN_DIMS     = [128, 64, 32]   # encoder layers; decoder mirrors this


# ── Feature matrix ────────────────────────────────────────────────────────────

def build_feature_matrix(df: pd.DataFrame):
    """
    Returns (X_scaled, feature_cols, scaler) for the anomaly detectors.
    Uses numeric + engineered one-hot columns only.
    """
    base_cols = [
        "event_id", "severity_value", "is_system",
        "process_depth", "hour_of_day", "day_of_week",
        "granted_access", "message_len",
    ]
    eid_cols     = [c for c in df.columns if c.startswith("eid_")]
    feature_cols = base_cols + eid_cols

    X      = df[feature_cols].fillna(0).astype(float).values
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    return X_scaled, feature_cols, scaler


# ── Model A: Isolation Forest ─────────────────────────────────────────────────

def run_isolation_forest(df: pd.DataFrame,
                         X_scaled: np.ndarray,
                         contamination: float = CONTAMINATION) -> pd.DataFrame:
    print("  [A] Training Isolation Forest …")
    iso = IsolationForest(
        contamination=contamination,
        n_estimators=200,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    df = df.copy()
    df["if_label"]        = iso.fit_predict(X_scaled)        # -1 = anomaly
    df["if_anomaly_score"] = iso.score_samples(X_scaled)     # lower = worse
    df["if_is_anomaly"]   = (df["if_label"] == -1).astype(int)
    n = df["if_is_anomaly"].sum()
    print(f"      IF anomalies: {n:,}  ({n / len(df) * 100:.2f}%)")
    return df


# ── Model B: One-Class SVM ────────────────────────────────────────────────────

def run_one_class_svm(df: pd.DataFrame,
                      X_scaled: np.ndarray,
                      nu: float = CONTAMINATION) -> pd.DataFrame:
    """
    One-Class SVM with RBF kernel.
    `nu` is an upper bound on the fraction of training errors (≈ contamination).
    Subsamples to 30 k rows for tractability (OCSVM is O(n²) in memory).
    """
    print("  [B] Training One-Class SVM …")
    n_fit = min(30_000, len(X_scaled))
    rng   = np.random.default_rng(RANDOM_STATE)
    idx   = rng.choice(len(X_scaled), n_fit, replace=False)

    ocsvm = OneClassSVM(kernel="rbf", nu=nu, gamma="scale")
    ocsvm.fit(X_scaled[idx])

    df = df.copy()
    df["ocsvm_label"]        = ocsvm.predict(X_scaled)       # -1 = anomaly
    df["ocsvm_anomaly_score"] = ocsvm.score_samples(X_scaled) # lower = worse
    df["ocsvm_is_anomaly"]   = (df["ocsvm_label"] == -1).astype(int)
    n = df["ocsvm_is_anomaly"].sum()
    print(f"      OCSVM anomalies: {n:,}  ({n / len(df) * 100:.2f}%)")
    return df


# ── Model C: Deep Autoencoder ─────────────────────────────────────────────────

class _Autoencoder(nn.Module):
    """Simple symmetric fully-connected autoencoder."""
    def __init__(self, input_dim: int, hidden_dims: list[int]):
        super().__init__()
        # Encoder
        enc_layers = []
        prev = input_dim
        for h in hidden_dims:
            enc_layers += [nn.Linear(prev, h), nn.BatchNorm1d(h), nn.ReLU()]
            prev = h
        self.encoder = nn.Sequential(*enc_layers)
        # Decoder (mirror)
        dec_layers = []
        for h in reversed(hidden_dims[:-1]):
            dec_layers += [nn.Linear(prev, h), nn.BatchNorm1d(h), nn.ReLU()]
            prev = h
        dec_layers.append(nn.Linear(prev, input_dim))
        self.decoder = nn.Sequential(*dec_layers)

    def forward(self, x):
        return self.decoder(self.encoder(x))


def run_autoencoder(df: pd.DataFrame,
                    X_scaled: np.ndarray,
                    epochs: int      = AE_EPOCHS,
                    batch_size: int  = AE_BATCH_SIZE,
                    lr: float        = AE_LR,
                    hidden_dims: list = AE_HIDDEN_DIMS,
                    contamination: float = CONTAMINATION) -> pd.DataFrame:
    """
    Trains a deep autoencoder on the full dataset (unsupervised).
    Anomaly score = per-sample mean squared reconstruction error.
    Events above the (1 - contamination) quantile are flagged anomalous.
    """
    print("  [C] Training Deep Autoencoder …")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"      Using device: {device}")

    X_tensor = torch.tensor(X_scaled, dtype=torch.float32).to(device)
    loader   = DataLoader(TensorDataset(X_tensor, X_tensor),
                          batch_size=batch_size, shuffle=True)

    model     = _Autoencoder(X_scaled.shape[1], hidden_dims).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()

    model.train()
    for epoch in range(epochs):
        total_loss = 0.0
        for x_batch, _ in loader:
            optimizer.zero_grad()
            loss = criterion(model(x_batch), x_batch)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * len(x_batch)
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"      Epoch [{epoch+1:02d}/{epochs}]  loss={total_loss/len(X_scaled):.6f}")

    # ── Compute reconstruction errors ─────────────────────────────────────────
    model.eval()
    all_errors = []
    with torch.no_grad():
        for i in range(0, len(X_tensor), batch_size):
            batch = X_tensor[i:i + batch_size]
            recon = model(batch)
            err   = ((recon - batch) ** 2).mean(dim=1).cpu().numpy()
            all_errors.append(err)
    recon_errors = np.concatenate(all_errors)

    threshold = np.quantile(recon_errors, 1 - contamination)

    df = df.copy()
    df["ae_anomaly_score"] = recon_errors                              # higher = worse (inverted from IF)
    df["ae_is_anomaly"]    = (recon_errors >= threshold).astype(int)
    n = df["ae_is_anomaly"].sum()
    print(f"      AE anomalies  : {n:,}  ({n / len(df) * 100:.2f}%)")
    return df


# ── UMAP visualisation ────────────────────────────────────────────────────────

def run_umap(X_scaled: np.ndarray, df: pd.DataFrame,
             sample_size: int = UMAP_SAMPLE):
    """UMAP 2-D projection of a random sample, coloured by IF anomaly score."""
    n   = min(sample_size, len(df))
    idx = np.random.default_rng(RANDOM_STATE).choice(len(df), n, replace=False)

    print(f"  UMAP on {n:,} samples …")
    reducer = umap.UMAP(
        n_components=2, n_neighbors=15, min_dist=0.1,
        metric="euclidean", random_state=RANDOM_STATE, low_memory=True,
    )
    emb = reducer.fit_transform(X_scaled[idx])

    plot_df = pd.DataFrame({
        "x"              : emb[:, 0],
        "y"              : emb[:, 1],
        "if_is_anomaly"  : df["if_is_anomaly"].iloc[idx].values,
        "if_anomaly_score": df["if_anomaly_score"].iloc[idx].values,
        "event_id"       : df["event_id"].iloc[idx].values,
        "hostname"       : df["hostname"].iloc[idx].values,
        "image_base"     : df["image_base"].iloc[idx].values,
    })

    fig = px.scatter(
        plot_df, x="x", y="y",
        color="if_anomaly_score",
        color_continuous_scale=["#e94560", "#f5a623", "#0f3460", "#16213e"],
        symbol="if_is_anomaly",
        hover_data=["event_id", "hostname", "image_base"],
        title="UMAP — Isolation Forest Anomaly Scores (Comparative Run)",
        template="plotly_dark",
        opacity=0.55,
        height=650,
        labels={"if_anomaly_score": "IF Score (lower = more anomalous)",
                "if_is_anomaly": "IF Anomaly"},
    )
    fig.update_traces(marker_size=3)
    fig.update_layout(
        font_family="Inter, sans-serif",
        title_font_size=18,
        coloraxis_colorbar_title="IF Score",
    )
    return fig


# ── Entry point ───────────────────────────────────────────────────────────────

def anomaly_stage(labeled_path: str    = LABELED_PARQUET,
                  anomalies_path: str  = ANOMALIES_PARQUET,
                  contamination: float = CONTAMINATION) -> pd.DataFrame:
    """
    End-to-end Stage 2 entry point.
    Runs all three anomaly detectors, saves anomalies.parquet, renders UMAP.
    """
    os.makedirs(os.path.dirname(anomalies_path), exist_ok=True)

    print("Loading labeled parquet …")
    df = pd.read_parquet(labeled_path)
    print(f"    Shape: {df.shape}")
    if "ground_truth" in df.columns:
        print(f"    Ground truth positives: {df['ground_truth'].sum():,} "
              f"({df['ground_truth'].mean() * 100:.1f}%)")

    X_scaled, feature_cols, _ = build_feature_matrix(df)

    print("\n── Running anomaly detectors ──────────────────────────────")
    df = run_isolation_forest(df, X_scaled, contamination)
    df = run_one_class_svm(df, X_scaled, nu=contamination)
    df = run_autoencoder(df, X_scaled, contamination=contamination)
    print("────────────────────────────────────────────────────────────")

    # Union anomaly flag: event is anomalous if ANY model flags it
    df["is_anomaly"] = (
        (df["if_is_anomaly"] == 1) |
        (df["ocsvm_is_anomaly"] == 1) |
        (df["ae_is_anomaly"] == 1)
    ).astype(int)

    n_union = df["is_anomaly"].sum()
    print(f"\n  Union anomalies (any model): {n_union:,}  ({n_union / len(df) * 100:.2f}%)")

    # Keep backward-compatible `anomaly_score` column (= IF score)
    df["anomaly_score"] = df["if_anomaly_score"]

    fig = run_umap(X_scaled, df)
    fig.show()

    anomalies_df = df[df["is_anomaly"] == 1].copy()
    anomalies_df.to_parquet(anomalies_path, index=False)
    print(f"\n  Anomalies saved → {anomalies_path}")

    # Summary table
    summary = (
        anomalies_df.groupby("event_id")
        .agg(count=("event_id", "size"),
             avg_if_score=("if_anomaly_score", "mean"),
             avg_ae_score=("ae_anomaly_score", "mean"),
             hosts=("hostname", "nunique"))
        .sort_values("avg_if_score")
        .head(15)
    )
    print("\nTop anomalous EventIDs (all models):\n", summary.to_string())

    return anomalies_df

In [ ]:
anomalies_df = anomaly_stage(
    labeled_path   = LABELED_PARQUET,
    anomalies_path = ANOMALIES_PARQUET,
    contamination  = CONTAMINATION,
)


In [ ]:
import pandas as pd

anomalies_df = pd.read_parquet(ANOMALIES_PARQUET)

# Top anomalous event types — all model scores
top = (
    anomalies_df.groupby("event_id")
    .agg(
        count=("event_id", "size"),
        avg_if_score=("if_anomaly_score", "mean"),
        avg_ae_score=("ae_anomaly_score", "mean"),
        unique_hosts=("hostname", "nunique"),
    )
    .sort_values("avg_if_score")
    .head(10)
)
display(top)


---
## Stage 3 — BERTopic (Semantic Topic Modelling)

Runs BERTopic on the anomalous event corpus:
1. **Clean** log text (strip GUIDs, hex, paths, timestamps)
2. **Embed** with `all-MiniLM-L6-v2` on GPU
3. **Cluster** with HDBSCAN
4. **Label** topics with c-TF-IDF keywords
5. **Visualise** — bar chart, inter-topic map, heatmap


In [ ]:
"""
Stage 3: BERTopic — Topic Modelling on Anomalous Events
Cleans log text, embeds with sentence-transformers, clusters with HDBSCAN,
and visualises topics interactively.
"""
import os
import re
import nltk
import pandas as pd
from tqdm.auto import tqdm

from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer

# ── Config ────────────────────────────────────────────────────────────────────
ANOMALIES_PARQUET = "/content/data/anomalies.parquet"
TOPICS_PARQUET    = "/content/data/anomalies_with_topics.parquet"
TOPIC_MODEL_DIR   = "/content/data/bertopic_model"
EMBED_MODEL       = "all-MiniLM-L6-v2"
MIN_CLUSTER_SIZE  = 15
TOP_N_WORDS       = 10
# ──────────────────────────────────────────────────────────────────────────────


# ── Text cleaning ─────────────────────────────────────────────────────────────
_GUID_RE      = re.compile(r"\{[0-9a-fA-F\-]{8,}\}")
_HEX_RE       = re.compile(r"\b0x[0-9a-fA-F]+\b")
_TS_RE        = re.compile(r"\d{4}-\d{2}-\d{2}[T ]\d{2}:\d{2}:\d{2}(?:\.\d+)?")
_PATH_RE      = re.compile(r"[A-Za-z]:\\(?:[^\s\r\n|,\\]+\\)*([^\s\r\n|,\\]+)")
_NUM_RE       = re.compile(r"\b\d+\b")
_WS_RE        = re.compile(r"\s+")


def _ensure_stopwords():
    try:
        from nltk.corpus import stopwords
        return set(stopwords.words("english"))
    except LookupError:
        nltk.download("stopwords", quiet=True)
        from nltk.corpus import stopwords
        return set(stopwords.words("english"))


STOP_WORDS = _ensure_stopwords()

# Windows / Sysmon noise words that add no semantic value
DOMAIN_NOISE = {
    "rulename", "utctime", "processguid", "processid", "image",
    "targetprocessid", "targetprocessguid", "sourcename", "channel",
    "keywords", "opcodevalue", "severityvalue", "eventreceivedtime",
    "sourcemodulename", "sourcemoduletype", "version", "task",
    "threadid", "recordnumber", "executionprocessid", "providerguid",
    "timestamp", "version", "none", "null", "true", "false",
}


def clean_log_text(text: str) -> str:
    """
    Strip technical noise (GUIDs, hex, paths, timestamps, numbers)
    and return a bag-of-meaningful-words string.
    """
    # Replace paths with just the exe/filename token
    text = _PATH_RE.sub(lambda m: " " + m.group(1).lower() + " ", text)
    text = _GUID_RE.sub(" ", text)
    text = _HEX_RE.sub(" hexval ", text)
    text = _TS_RE.sub(" ", text)
    text = _NUM_RE.sub(" ", text)
    # Keep only alpha
    text = re.sub(r"[^a-zA-Z\s]", " ", text)
    text = _WS_RE.sub(" ", text).strip().lower()

    tokens = [
        w for w in text.split()
        if w not in STOP_WORDS
        and w not in DOMAIN_NOISE
        and len(w) > 2
    ]
    return " ".join(tokens) if tokens else "unknown_event"


# ── BERTopic pipeline ─────────────────────────────────────────────────────────

def build_topic_model() -> BERTopic:
    umap_model = UMAP(
        n_neighbors=15,
        n_components=5,
        min_dist=0.0,
        metric="cosine",
        random_state=42,
        low_memory=True,
    )
    hdbscan_model = HDBSCAN(
        min_cluster_size=MIN_CLUSTER_SIZE,
        metric="euclidean",
        cluster_selection_method="eom",
        prediction_data=True,
    )
    vectorizer = CountVectorizer(
        stop_words="english",
        min_df=2,
        ngram_range=(1, 2),
        max_features=10_000,
    )
    embedding_model = SentenceTransformer(EMBED_MODEL)

    return BERTopic(
        embedding_model=embedding_model,
        umap_model=umap_model,
        hdbscan_model=hdbscan_model,
        vectorizer_model=vectorizer,
        top_n_words=TOP_N_WORDS,
        calculate_probabilities=True,
        verbose=True,
    )


def topic_stage(anomalies_path: str = ANOMALIES_PARQUET,
                topics_path: str    = TOPICS_PARQUET,
                model_dir: str      = TOPIC_MODEL_DIR) -> tuple[pd.DataFrame, BERTopic]:
    """
    End-to-end Stage 3 entry point.
    Returns (annotated_df, topic_model).
    """
    os.makedirs(os.path.dirname(topics_path), exist_ok=True)
    os.makedirs(model_dir, exist_ok=True)

    print("Loading anomalies...")
    df = pd.read_parquet(anomalies_path)
    print(f"    Shape: {df.shape}")

    # ── Prepare corpus ────────────────────────────────────────────────────────
    print("Cleaning log text...")

    def _col(name: str) -> pd.Series:
        """Safely retrieve a column; return empty strings if absent."""
        return df[name].fillna("") if name in df.columns else pd.Series("", index=df.index)

    corpus_raw = (
        _col("message") + " " +
        _col("image_base") + " " +
        _col("target_image_base") + " " +
        _col("target_object").apply(
            lambda x: x.split("\\")[-1].lower() if isinstance(x, str) and x else ""
        )
    )
    docs = corpus_raw.apply(clean_log_text).tolist()
    print(f"    Corpus size: {len(docs):,} documents")
    print(f"    Sample doc : {docs[0][:120]}")

    # ── Fit BERTopic ──────────────────────────────────────────────────────────
    print("\nFitting BERTopic...")
    topic_model = build_topic_model()
    topics, probs = topic_model.fit_transform(docs)

    df = df.copy()
    df["topic"]       = topics
    df["topic_prob"]  = [float(p.max()) if hasattr(p, "max") else float(p)
                         for p in probs]

    n_topics = len(set(topics)) - (1 if -1 in topics else 0)
    print(f"\nDiscovered {n_topics} topics (topic -1 = noise/outliers)")

    # ── Topic info ────────────────────────────────────────────────────────────
    topic_info = topic_model.get_topic_info()
    print("\nTop topics:\n", topic_info.head(12).to_string(index=False))

    # ── Visualisations ────────────────────────────────────────────────────────
    print("\nGenerating visualisations...")

    fig_bar = topic_model.visualize_barchart(
        top_n_topics=min(12, n_topics), n_words=8
    )
    fig_bar.update_layout(template="plotly_dark",
                          title="Security Event Topics — Top Keywords")
    fig_bar.show()

    if n_topics >= 2:
        fig_map = topic_model.visualize_topics()
        fig_map.update_layout(template="plotly_dark",
                              title="Inter-topic Distance Map")
        fig_map.show()

        fig_heat = topic_model.visualize_heatmap()
        fig_heat.update_layout(template="plotly_dark",
                               title="Topic Similarity Heatmap")
        fig_heat.show()

    # ── Save ──────────────────────────────────────────────────────────────────
    df.to_parquet(topics_path, index=False)
    topic_model.save(os.path.join(model_dir, "model.pkl"))
    print(f"\nSaved annotated parquet -> {topics_path}")
    print(f"Saved BERTopic model    -> {model_dir}")

    return df, topic_model


def get_topic_summary(topic_model: BERTopic, top_n: int = 15) -> dict:
    """
    Returns {topic_id: 'keyword1, keyword2, …'} for the RAG query builder.
    """
    summary = {}
    for tid in topic_model.get_topic_info()["Topic"].tolist():
        if tid == -1:
            continue
        words = topic_model.get_topic(tid)
        if words:
            summary[tid] = ", ".join([w for w, _ in words[:top_n]])
    return summary

In [ ]:
df_a, topic_model = topic_stage(
    anomalies_path = ANOMALIES_PARQUET,
    topics_path    = TOPICS_PARQUET,
    model_dir      = TOPIC_MODEL_DIR,
)


In [ ]:
import pandas as pd

topics_df = pd.read_parquet(TOPICS_PARQUET)
kw_map    = get_topic_summary(topic_model, top_n=8)

topics_df["topic_keywords"] = topics_df["topic"].map(
    lambda t: kw_map.get(t, "unknown")
)
topics_df.to_parquet(TOPICS_PARQUET, index=False)

print("Topic keyword map (first 8 topics):")
for tid, kw in list(kw_map.items())[:8]:
    count = (topics_df["topic"] == tid).sum()
    print(f"  Topic {tid:3d} ({count:5,} events): {kw}")


---
## Stage 4a — Build RAG Knowledge Base (ChromaDB)

Downloads and indexes **5 cybersecurity knowledge sources**:

| Collection | Source | Content |
|---|---|---|
| `mitre_attack` | MITRE ATT&CK v14 | ~700 techniques + descriptions |
| `mitre_d3fend` | MITRE D3FEND | Defensive countermeasures |
| `mitre_car` | MITRE CAR | Detection analytics & pseudocode |
| `cisa_kev` | CISA KEV | Known exploited CVEs + required actions |
| `sigma_rules` | SigmaHQ | 3000+ YAML detection rules |

> This cell takes **5–15 min** (cloning Sigma is the slow part). Run once; ChromaDB persists to disk.


In [ ]:
"""
Stage 4a: RAG Knowledge Base Builder
Downloads and indexes 5 cybersecurity knowledge sources into ChromaDB.
Sources: MITRE ATT&CK, MITRE D3FEND, MITRE CAR, CISA KEV, SigmaHQ Rules
"""
import os, re, json, subprocess
import requests
import chromadb
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

# ── Config ────────────────────────────────────────────────────────────────────
CHROMA_DIR   = "/content/data/chroma_db"
EMBED_MODEL  = "all-MiniLM-L6-v2"
BATCH_SIZE   = 128
KB_DIR       = "/content/data/kb_raw"
# ──────────────────────────────────────────────────────────────────────────────

_emb_model: SentenceTransformer = None

def _get_embedder():
    global _emb_model
    if _emb_model is None:
        print("Loading embedding model...")
        _emb_model = SentenceTransformer(EMBED_MODEL)
    return _emb_model


def _get_client():
    os.makedirs(CHROMA_DIR, exist_ok=True)
    return chromadb.PersistentClient(path=CHROMA_DIR)


def _upsert_collection(client: chromadb.Client,
                       name: str,
                       docs: list[str],
                       ids: list[str],
                       metas: list[dict]):
    """Create or replace a ChromaDB collection and batch-embed docs."""
    try:
        client.delete_collection(name)
    except Exception:
        pass
    col = client.create_collection(name)
    embedder = _get_embedder()

    for i in tqdm(range(0, len(docs), BATCH_SIZE), desc=f"  Indexing {name}"):
        bd = docs[i:i+BATCH_SIZE]
        bi = ids[i:i+BATCH_SIZE]
        bm = metas[i:i+BATCH_SIZE]
        emb = embedder.encode(bd, show_progress_bar=False).tolist()
        col.add(documents=bd, ids=bi, embeddings=emb, metadatas=bm)

    print(f"  {name}: {col.count():,} docs indexed")
    return col


# ── Source 1: MITRE ATT&CK ───────────────────────────────────────────────────

def load_mitre_attack(client):
    print("\nMITRE ATT&CK v14...")
    url = ("https://raw.githubusercontent.com/mitre/cti/master/"
           "enterprise-attack/enterprise-attack.json")
    data = requests.get(url, timeout=120).json()

    docs, ids, metas = [], [], []
    for obj in data["objects"]:
        if obj.get("type") != "attack-pattern":
            continue
        tech_id, tactic = "", ""
        for ref in obj.get("external_references", []):
            if ref.get("source_name") == "mitre-attack":
                tech_id = ref.get("external_id", "")
        for phase in obj.get("kill_chain_phases", []):
            tactic = phase.get("phase_name", "")
        name = obj.get("name", "")
        desc = obj.get("description", "")[:1800]
        text = f"ATT&CK {tech_id} — {name}\nTactic: {tactic}\n{desc}"
        uid  = f"attack_{tech_id}_{obj['id'][-6:]}"
        docs.append(text); ids.append(uid)
        metas.append({"source": "mitre_attack", "tech_id": tech_id,
                      "tactic": tactic, "name": name})

    _upsert_collection(client, "mitre_attack", docs, ids, metas)


# ── Source 2: MITRE D3FEND ───────────────────────────────────────────────────

def load_d3fend(client):
    print("\nMITRE D3FEND...")
    url = "https://d3fend.mitre.org/api/technique/all.json"
    try:
        data = requests.get(url, timeout=60).json()
        techniques = data.get("techniques") or data.get("data") or []
    except Exception as e:
        print(f"  D3FEND fetch failed: {e}")
        return

    docs, ids, metas = [], [], []
    for t in techniques:
        tid   = t.get("id") or t.get("d3f_id") or "unknown"
        label = t.get("label") or t.get("name") or ""
        desc  = t.get("definition") or t.get("description") or ""
        text  = f"D3FEND {tid} — {label}\n{desc}"[:2000]
        uid   = f"d3fend_{re.sub(r'[^a-zA-Z0-9_]', '_', tid)}"
        docs.append(text); ids.append(uid)
        metas.append({"source": "d3fend", "d3fend_id": tid, "name": label})

    if docs:
        _upsert_collection(client, "mitre_d3fend", docs, ids, metas)
    else:
        print("  D3FEND returned 0 usable techniques.")


# ── Source 3: MITRE CAR ──────────────────────────────────────────────────────

def load_car(client):
    print("\nMITRE CAR analytics...")
    car_dir = os.path.join(KB_DIR, "car")
    if not os.path.exists(car_dir):
        subprocess.run(
            ["git", "clone", "--depth=1",
             "https://github.com/mitre-attack/car.git", car_dir],
            check=True, capture_output=True,
        )

    import yaml
    docs, ids, metas = [], [], []
    analytics_dir = os.path.join(car_dir, "analytics")
    if not os.path.exists(analytics_dir):
        print("  CAR analytics directory not found.")
        return

    for fname in os.listdir(analytics_dir):
        if not (fname.endswith(".yaml") or fname.endswith(".yml")):
            continue
        try:
            with open(os.path.join(analytics_dir, fname), "r", encoding="utf-8") as f:
                obj = yaml.safe_load(f)
            title = obj.get("title", fname)
            desc  = obj.get("description", "")
            impl  = " ".join(
                str(i.get("code", ""))
                for i in (obj.get("implementations") or [])
                if isinstance(i, dict)
            )
            text = f"CAR — {title}\n{desc}\nDetection logic: {impl}"[:2000]
            uid  = f"car_{fname.replace('.yaml','').replace('.yml','')[:60]}"
            docs.append(text); ids.append(uid)
            metas.append({"source": "mitre_car", "title": title, "file": fname})
        except Exception:
            continue

    if docs:
        _upsert_collection(client, "mitre_car", docs, ids, metas)


# ── Source 4: CISA KEV ───────────────────────────────────────────────────────

def load_cisa_kev(client):
    print("\nCISA Known Exploited Vulnerabilities...")
    url = ("https://www.cisa.gov/sites/default/files/feeds/"
           "known_exploited_vulnerabilities.json")
    data = requests.get(url, timeout=60).json()

    docs, ids, metas = [], [], []
    for v in data.get("vulnerabilities", []):
        cve  = v.get("cveID", "unknown")
        text = (
            f"CVE: {cve} | Vendor: {v.get('vendorProject','')} | "
            f"Product: {v.get('product','')} | "
            f"Vulnerability: {v.get('vulnerabilityName','')} | "
            f"Required Action: {v.get('requiredAction','')} | "
            f"Due Date: {v.get('dueDate','')} | "
            f"Notes: {v.get('notes','')}"
        )[:2000]
        uid = f"kev_{cve}"
        docs.append(text); ids.append(uid)
        metas.append({"source": "cisa_kev", "cve_id": cve,
                      "product": v.get("product", "")})

    _upsert_collection(client, "cisa_kev", docs, ids, metas)


# ── Source 5: SigmaHQ Rules ──────────────────────────────────────────────────

def load_sigma(client):
    print("\nSigmaHQ detection rules (sparse clone)...")
    sigma_dir = os.path.join(KB_DIR, "sigma")
    if not os.path.exists(sigma_dir):
        subprocess.run(
            ["git", "clone", "--depth=1", "--filter=blob:none", "--sparse",
             "https://github.com/SigmaHQ/sigma.git", sigma_dir],
            check=True, capture_output=True,
        )
        subprocess.run(
            ["git", "sparse-checkout", "set", "rules/"],
            cwd=sigma_dir, check=True, capture_output=True,
        )

    import yaml
    docs, ids, metas, seen = [], [], [], set()
    rules_dir = os.path.join(sigma_dir, "rules")

    for root, _, files in os.walk(rules_dir):
        for fname in files:
            if not fname.endswith(".yml"):
                continue
            uid = f"sigma_{fname[:60]}"
            if uid in seen:
                uid += f"_{len(seen)}"
            seen.add(uid)
            try:
                with open(os.path.join(root, fname), "r",
                          encoding="utf-8", errors="replace") as f:
                    raw = f.read()
                obj   = yaml.safe_load(raw) or {}
                title = obj.get("title", fname)
                desc  = obj.get("description", "")
                tags  = ", ".join(obj.get("tags") or [])
                detect = str(obj.get("detection") or "")
                text = (
                    f"Sigma Rule: {title}\n"
                    f"Tags: {tags}\n"
                    f"Description: {desc}\n"
                    f"Detection: {detect}"
                )[:2000]
                docs.append(text); ids.append(uid)
                metas.append({"source": "sigma", "title": title,
                               "file": fname, "tags": tags})
            except Exception:
                continue

    if docs:
        _upsert_collection(client, "sigma_rules", docs, ids, metas)


# ── Source 6: MSRC CVRF ──────────────────────────────────────────────────────

def load_msrc_cvrf(client):
    print("\nMicrosoft Security Update Summaries (MSRC CVRF) (2023+)...")
    url = "https://api.msrc.microsoft.com/cvrf/v3.0/updates"
    try:
        data = requests.get(url, timeout=60).json()
        updates = data.get("value", [])
    except Exception as e:
        print(f"  MSRC fetch failed: {e}")
        return

    docs, ids, metas = [], [], []
    for u in updates:
        id_str = u.get("ID", "")
        if not id_str:
            continue
        
        # Limit to 2023 onwards
        year_str = id_str[:4]
        try:
            if int(year_str) < 2023:
                continue
        except ValueError:
            continue

        doc_title = u.get("DocumentTitle", "")
        init_date = u.get("InitialReleaseDate", "")
        cvrf_url = u.get("CvrfUrl", "")

        text = (
            f"MSRC Update: {id_str} | "
            f"Title: {doc_title} | "
            f"Release Date: {init_date} | "
            f"CVRF URL: {cvrf_url}"
        )[:2000]

        uid = f"msrc_{id_str}"
        docs.append(text)
        ids.append(uid)
        metas.append({"source": "msrc_cvrf", "update_id": id_str, "title": doc_title})

    if docs:
        _upsert_collection(client, "msrc_cvrf", docs, ids, metas)


# ── Entry point ───────────────────────────────────────────────────────────────

def build_knowledge_base():
    """Download and index all knowledge sources."""
    os.makedirs(KB_DIR, exist_ok=True)
    client = _get_client()

    load_mitre_attack(client)
    load_d3fend(client)
    load_car(client)
    load_cisa_kev(client)
    load_sigma(client)
    load_msrc_cvrf(client)

    print("\nKnowledge base complete.")
    print(f"    Collections: {[c.name for c in client.list_collections()]}")
    return client


def query_all_collections(client: chromadb.Client,
                          query: str,
                          top_k: int = 5) -> str:
    """
    Semantic search across all indexed collections.
    Returns a single concatenated context string.
    """
    embedder = _get_embedder()
    q_emb = embedder.encode([query]).tolist()
    parts  = []

    for col in client.list_collections():
        try:
            n = min(top_k, col.count())
            if n == 0:
                continue
            res = col.query(query_embeddings=q_emb, n_results=n)
            for doc, meta in zip(res["documents"][0], res["metadatas"][0]):
                src = meta.get("source", col.name).upper()
                parts.append(f"[{src}]\n{doc[:600]}")
        except Exception as e:
            print(f"  {col.name}: {e}")

    return "\n\n---\n\n".join(parts)

In [ ]:
chroma_client = build_knowledge_base()


In [ ]:
# Verify collections
chroma_client = _get_client()
print("ChromaDB collections:")
for col in chroma_client.list_collections():
    print(f"  {col.name:20s} → {col.count():,} docs")

# Quick test query
test_ctx = query_all_collections(chroma_client, "lsass process access credential dump", top_k=2)
print("\nTest query (lsass credential dump):")
print(test_ctx[:800])


---



In [ ]:
# Install Ollama
!sudo apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

# Start the server in the background
import subprocess
import time
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(4)

# Pull the model (you will see the progress bar here!)
!ollama pull llama3


---
## Stage 4b — LLM Threat Analysis (DSPy + Ollama + RAG)

### What happens per anomalous event:
1. **Build context** string from event fields
2. **Retrieve** relevant docs from all ChromaDB collections (semantic search)
3. **DSPy `ChainOfThought`** reasons step-by-step over context + retrieved docs
4. **Ollama `llama3`** generates structured output:
   - `threat_analysis` — what the threat is
   - `mitre_technique` — ATT&CK ID + name
   - `remediation_steps` — numbered list
   - `severity_rating` — Critical / High / Medium / Low

> **DSPy `BootstrapFewShot`** auto-optimises prompt selection using 2 APT29 labelled examples.


In [ ]:
"""
Stage 4b: LLM Analysis — DSPy + Ollama
Defines the DSPy signature, ChainOfThought module, and BootstrapFewShot
optimizer. Runs threat analysis on the top anomalous events per BERTopic cluster.
"""
import os, json
import pandas as pd
import dspy
import chromadb
from IPython.display import display, Markdown

try:
    from pipeline.stage4a_rag_kb import query_all_collections, _get_client, _get_embedder
except ImportError:
    pass  # In notebook mode, these functions are already in the global namespace

# ── Config ────────────────────────────────────────────────────────────────────
TOPICS_PARQUET  = "/content/data/anomalies_with_topics.parquet"
RESULTS_JSON    = "/content/data/llm_results.json"
OLLAMA_MODEL    = "llama3"           # swap to mistral / phi3 if preferred
OLLAMA_BASE_URL = "http://localhost:11434"
TOP_N_TOPICS    = 12                 # topics to analyse
EVENTS_PER_TOPIC = 3                 # worst-scoring events per topic
RAG_TOP_K       = 5
# ──────────────────────────────────────────────────────────────────────────────


# ── Ollama bootstrap ──────────────────────────────────────────────────────────

def configure_dspy(model: str = OLLAMA_MODEL):
    """Wire DSPy to the local Ollama endpoint."""
    lm = dspy.LM(
        model=f"ollama_chat/{model}",
        api_base=OLLAMA_BASE_URL,
        api_key="ollama",
        temperature=0.15,
        max_tokens=1024,
    )
    dspy.configure(lm=lm)
    print(f"  DSPy configured -> ollama/{model}")
    return lm


# ── DSPy Signature ────────────────────────────────────────────────────────────

class ThreatAnalysis(dspy.Signature):
    """
    You are a senior threat-intelligence analyst.
    Given a Windows security anomaly, related topic keywords, and retrieved
    threat-intel context, produce a structured analysis.
    """
    anomaly_context: str  = dspy.InputField(
        desc="EventID, process image, hostname, account, granted access, "
             "and the raw Sysmon message snippet."
    )
    topic_keywords: str   = dspy.InputField(
        desc="BERTopic cluster keywords that describe the group this event belongs to."
    )
    retrieved_docs: str   = dspy.InputField(
        desc="Relevant passages from MITRE ATT&CK, D3FEND, CAR, CISA KEV, "
             "and Sigma rules retrieved via semantic search."
    )

    threat_analysis: str    = dspy.OutputField(
        desc="1–3 sentence analysis of the likely threat this anomaly represents."
    )
    mitre_technique: str    = dspy.OutputField(
        desc="Most applicable MITRE ATT&CK technique, e.g. 'T1055 - Process Injection'."
    )
    remediation_steps: str  = dspy.OutputField(
        desc="Numbered list of 3–5 concrete detection or remediation steps."
    )
    severity_rating: str    = dspy.OutputField(
        desc="One of: Critical / High / Medium / Low — with a one-sentence rationale."
    )


# ── DSPy Module ───────────────────────────────────────────────────────────────

class SecurityAnalyzer(dspy.Module):
    def __init__(self):
        super().__init__()
        self.analyze = dspy.ChainOfThought(ThreatAnalysis)

    def forward(self, anomaly_context: str,
                topic_keywords: str,
                retrieved_docs: str) -> dspy.Prediction:
        return self.analyze(
            anomaly_context=anomaly_context,
            topic_keywords=topic_keywords,
            retrieved_docs=retrieved_docs,
        )


# ── Few-shot examples for BootstrapFewShot ───────────────────────────────────

FEW_SHOT_EXAMPLES = [
    dspy.Example(
        anomaly_context=(
            "EventID: 10 | Process: C:\\Windows\\System32\\lsass.exe | "
            "Target: lsass.exe | GrantedAccess: 0x1FFFFF | "
            "Message: Process accessed lsass with full handle rights."
        ),
        topic_keywords="lsass, credential, access, memory, dump, mimikatz, process",
        retrieved_docs="[MITRE_ATTACK] T1003.001 - LSASS Memory: Adversaries may "
                       "attempt to access credential material stored in LSASS.",
        threat_analysis=(
            "This event strongly indicates credential dumping via direct LSASS "
            "memory access, consistent with tools like Mimikatz or ProcDump. "
            "Full handle rights (0x1FFFFF) are rarely required by legitimate processes."
        ),
        mitre_technique="T1003.001 - OS Credential Dumping: LSASS Memory",
        remediation_steps=(
            "1. Enable Credential Guard (Windows 10/11).\n"
            "2. Restrict LSASS access via Protected Process Light (PPL).\n"
            "3. Alert on GrantedAccess 0x1FFFFF targeting lsass.exe.\n"
            "4. Deploy Sigma rule 'win_lsass_access_non_system_account'.\n"
            "5. Review process lineage for the accessing process."
        ),
        severity_rating="Critical — Direct credential theft enables lateral movement.",
    ).with_inputs("anomaly_context", "topic_keywords", "retrieved_docs"),

    dspy.Example(
        anomaly_context=(
            "EventID: 13 | Process: reg.exe | "
            "TargetObject: HKLM\\SOFTWARE\\Microsoft\\Windows\\CurrentVersion\\Run\\backdoor | "
            "Message: Registry value set for persistence."
        ),
        topic_keywords="registry, persistence, run, key, startup, autorun",
        retrieved_docs="[MITRE_ATTACK] T1547.001 - Registry Run Keys: Adversaries "
                       "may achieve persistence by adding a program to a Run key.",
        threat_analysis=(
            "A new Run key was written by reg.exe, a classic persistence mechanism. "
            "The key name 'backdoor' is highly suspicious and warrants immediate review."
        ),
        mitre_technique="T1547.001 - Boot or Logon Autostart: Registry Run Keys",
        remediation_steps=(
            "1. Remove the malicious Run key immediately.\n"
            "2. Alert on unexpected writes to HKLM\\SOFTWARE\\...\\Run\\.\n"
            "3. Audit reg.exe invocations not launched by administrators.\n"
            "4. Use Sigma rule 'win_registry_run_key_modification'.\n"
            "5. Investigate the parent process that spawned reg.exe."
        ),
        severity_rating="High — Persistence mechanism allows re-infection after reboot.",
    ).with_inputs("anomaly_context", "topic_keywords", "retrieved_docs"),
]


def optimize_analyzer(analyzer: SecurityAnalyzer,
                      examples: list = FEW_SHOT_EXAMPLES) -> SecurityAnalyzer:
    """Run BootstrapFewShot to auto-select best prompts."""
    print("  Running DSPy BootstrapFewShot optimisation...")
    optimizer = dspy.BootstrapFewShot(max_bootstrapped_demos=2,
                                      max_labeled_demos=2)
    # Metric: response is non-empty (adjust with a real eval if labels exist)
    def metric(example, pred, trace=None):
        return (bool(pred.threat_analysis) and
                bool(pred.mitre_technique) and
                bool(pred.remediation_steps))

    optimized = optimizer.compile(analyzer, trainset=examples, metric=metric)
    print("  Optimisation complete.")
    return optimized


# ── Context builder ───────────────────────────────────────────────────────────

def build_anomaly_context(row: pd.Series) -> str:
    ga = row.get("granted_access", 0)
    ga_hex = hex(int(ga)) if ga else "N/A"
    return (
        f"EventID: {row['event_id']} | "
        f"Hostname: {row['hostname']} | "
        f"Process: {row['image']} | "
        f"TargetImage: {row.get('target_image', '')} | "
        f"TargetObject: {str(row.get('target_object', ''))[:80]} | "
        f"Account: {row['account_name']} ({row['domain']}) | "
        f"GrantedAccess: {ga_hex} | "
        f"IsolationForest score: {row.get('anomaly_score', 'N/A'):.4f} | "
        f"Message: {str(row.get('message', ''))[:400]}"
    )


# ── Main analysis loop ────────────────────────────────────────────────────────

def display_result(result: dict, idx: int):
    md = f"""
---
### Analysis #{idx+1} — Topic {result['topic_id']}
**Event:** `{result['event_id']}` on `{result['hostname']}`
**Topic keywords:** _{result['topic_keywords']}_

**Threat Analysis**
{result['threat_analysis']}

**MITRE ATT&CK Technique**
`{result['mitre_technique']}`

**Remediation Steps**
{result['remediation_steps']}

**Severity:** {result['severity_rating']}
---
"""
    display(Markdown(md))


def llm_analysis_stage(topics_path: str    = TOPICS_PARQUET,
                       results_path: str   = RESULTS_JSON,
                       skip_optimize: bool = False) -> list[dict]:
    """
    End-to-end Stage 4b entry point.
    Returns list of result dicts, also saved to JSON.
    """
    # ── Setup ─────────────────────────────────────────────────────────────────
    configure_dspy(OLLAMA_MODEL)

    client   = _get_client()
    analyzer = SecurityAnalyzer()

    if not skip_optimize:
        try:
            analyzer = optimize_analyzer(analyzer)
        except Exception as e:
            print(f"  Optimisation skipped: {e}")

    # ── Load data ─────────────────────────────────────────────────────────────
    print(f"\n  Loading {topics_path}...")
    df = pd.read_parquet(topics_path)
    valid_topics = sorted(
        [t for t in df["topic"].unique() if t != -1],
        key=lambda t: df[df["topic"] == t]["anomaly_score"].mean()
    )[:TOP_N_TOPICS]

    print(f"  Analysing {len(valid_topics)} topics × {EVENTS_PER_TOPIC} events each...\n")

    results = []

    for topic_id in valid_topics:
        t_df = df[df["topic"] == topic_id].nsmallest(EVENTS_PER_TOPIC, "anomaly_score")

        # Get topic keywords from BERTopic (stored in df if available, else from model)
        topic_kw = df[df["topic"] == topic_id]["topic_keywords"].iloc[0] \
                   if "topic_keywords" in df.columns else f"topic_{topic_id}"

        for _, row in t_df.iterrows():
            ctx   = build_anomaly_context(row)
            query = f"{ctx} {topic_kw}"
            docs  = query_all_collections(client, query, top_k=RAG_TOP_K)

            try:
                pred = analyzer(
                    anomaly_context=ctx,
                    topic_keywords=topic_kw,
                    retrieved_docs=docs,
                )
                rec = {
                    "topic_id"         : int(topic_id),
                    "topic_keywords"   : topic_kw,
                    "event_id"         : int(row["event_id"]),
                    "hostname"         : row["hostname"],
                    "anomaly_score"    : float(row.get("anomaly_score", 0)),
                    "threat_analysis"  : pred.threat_analysis,
                    "mitre_technique"  : pred.mitre_technique,
                    "remediation_steps": pred.remediation_steps,
                    "severity_rating"  : pred.severity_rating,
                }
                results.append(rec)
                display_result(rec, len(results) - 1)

            except Exception as e:
                print(f"  Topic {topic_id} event failed: {e}")

    # ── Save results ──────────────────────────────────────────────────────────
    os.makedirs(os.path.dirname(results_path), exist_ok=True)
    with open(results_path, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2)
    print(f"\n  Results saved -> {results_path}")
    print(f"    Total analyses: {len(results)}")
    return results

In [ ]:
results = llm_analysis_stage(
    topics_path   = TOPICS_PARQUET,
    results_path  = RESULTS_JSON,
    skip_optimize = False,   # set True to skip BootstrapFewShot (faster)
)


---
## Stage 5 — Quantitative ML Evaluation

Compares the three anomaly detectors against the ground truth labels (Stage 0)
using the following metrics:

| Metric | Description |
|---|---|
| **ROC-AUC** | Threshold-free discrimination ability |
| **PR-AUC** | Better than ROC-AUC for imbalanced security log data |
| **F1-Score** | Harmonic mean of Precision and Recall at optimal threshold |
| **Precision** | Fraction of flagged events that are actually malicious |
| **Recall** | Fraction of malicious events that were caught |
| **FPR** | False Positive Rate — the SOC analyst burden metric |

Generates thesis-quality figures: ROC curves, PR curves, and confusion matrices.


In [ ]:
"""
Stage 5: Quantitative ML Evaluation
Loads anomalies.parquet (which contains ground_truth labels from Stage 0
and anomaly scores from all three models in Stage 2).

Generates thesis-quality evaluation artefacts:
  ├── figures/roc_curve_comparison.png        — Overlaid ROC curves
  ├── figures/pr_curve_comparison.png         — Overlaid Precision-Recall curves
  ├── figures/confusion_matrices.png          — 3 × confusion matrices at optimal threshold
  └── metrics_report.json                     — All numeric scores

Metrics computed per model
──────────────────────────
  • ROC-AUC   — area under the ROC curve (threshold-free discrimination)
  • PR-AUC    — area under the Precision-Recall curve (better for imbalanced data)
  • F1 @ opt  — F1 at the threshold that maximises it
  • Precision @ opt
  • Recall    @ opt
  • FPR       @ opt  — False Positive Rate (SOC analyst burden)
"""
import os
import json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")          # non-interactive backend for Colab/headless
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.metrics import (
    roc_curve, auc, precision_recall_curve, average_precision_score,
    confusion_matrix, f1_score, precision_score, recall_score,
)

ANOMALIES_PARQUET  = "/content/data/anomalies.parquet"
FIGURES_DIR        = "/content/data/figures"
METRICS_JSON       = "/content/data/metrics_report.json"

# Model display names and their score columns.
# NOTE: IF and OCSVM scores are "lower = more anomalous", so we negate them.
MODELS = {
    "Isolation Forest": {
        "score_col"  : "if_anomaly_score",
        "flag_col"   : "if_is_anomaly",
        "negate"     : True,          # negate so higher = more anomalous for ROC
        "color"      : "#e94560",
        "linestyle"  : "-",
    },
    "One-Class SVM": {
        "score_col"  : "ocsvm_anomaly_score",
        "flag_col"   : "ocsvm_is_anomaly",
        "negate"     : True,
        "color"      : "#f5a623",
        "linestyle"  : "--",
    },
    "Deep Autoencoder": {
        "score_col"  : "ae_anomaly_score",
        "flag_col"   : "ae_is_anomaly",
        "negate"     : False,         # AE: higher reconstruction error = more anomalous
        "color"      : "#00b4d8",
        "linestyle"  : "-.",
    },
}

plt.style.use("dark_background")
FONT = {"family": "DejaVu Sans", "size": 12}
plt.rc("font", **FONT)


# ── Helpers ───────────────────────────────────────────────────────────────────

def _optimal_threshold_metrics(y_true, y_score):
    """
    Find the threshold that maximises F1.
    Returns dict with precision, recall, f1, fpr at that threshold.
    """
    prec, rec, thresholds = precision_recall_curve(y_true, y_score)
    # avoid div-by-zero
    f1s = np.where((prec + rec) == 0, 0, 2 * prec * rec / (prec + rec))
    best_idx = np.argmax(f1s)
    best_thr = thresholds[best_idx] if best_idx < len(thresholds) else thresholds[-1]

    y_pred = (y_score >= best_thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0

    return {
        "threshold" : float(best_thr),
        "precision" : float(precision_score(y_true, y_pred, zero_division=0)),
        "recall"    : float(recall_score(y_true, y_pred, zero_division=0)),
        "f1"        : float(f1_score(y_true, y_pred, zero_division=0)),
        "fpr"       : float(fpr),
        "cm"        : confusion_matrix(y_true, y_pred, labels=[0, 1]).tolist(),
    }


# ── Plot 1: ROC Curves ────────────────────────────────────────────────────────

def plot_roc_curves(y_true, model_scores: dict, out_path: str):
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.plot([0, 1], [0, 1], "w--", linewidth=1, label="Random (AUC = 0.50)")

    for name, (score, cfg) in model_scores.items():
        fpr, tpr, _ = roc_curve(y_true, score)
        roc_auc     = auc(fpr, tpr)
        ax.plot(fpr, tpr,
                color=cfg["color"], linestyle=cfg["linestyle"], linewidth=2.5,
                label=f"{name}  (AUC = {roc_auc:.3f})")

    ax.set_xlabel("False Positive Rate", fontsize=13)
    ax.set_ylabel("True Positive Rate (Recall)", fontsize=13)
    ax.set_title("ROC Curve Comparison — All Anomaly Detectors", fontsize=15, pad=14)
    ax.legend(loc="lower right", fontsize=11)
    ax.set_xlim(0, 1); ax.set_ylim(0, 1.02)
    ax.grid(alpha=0.25)
    fig.tight_layout()
    fig.savefig(out_path, dpi=150)
    plt.close(fig)
    print(f"  Saved → {out_path}")


# ── Plot 2: Precision-Recall Curves ──────────────────────────────────────────

def plot_pr_curves(y_true, model_scores: dict, out_path: str):
    baseline = y_true.mean()
    fig, ax  = plt.subplots(figsize=(8, 6))
    ax.axhline(baseline, color="white", linestyle="--", linewidth=1,
               label=f"Random (AP = {baseline:.3f})")

    for name, (score, cfg) in model_scores.items():
        prec, rec, _ = precision_recall_curve(y_true, score)
        ap           = average_precision_score(y_true, score)
        ax.plot(rec, prec,
                color=cfg["color"], linestyle=cfg["linestyle"], linewidth=2.5,
                label=f"{name}  (AP = {ap:.3f})")

    ax.set_xlabel("Recall", fontsize=13)
    ax.set_ylabel("Precision", fontsize=13)
    ax.set_title("Precision-Recall Curve Comparison", fontsize=15, pad=14)
    ax.legend(loc="upper right", fontsize=11)
    ax.set_xlim(0, 1); ax.set_ylim(0, 1.02)
    ax.grid(alpha=0.25)
    fig.tight_layout()
    fig.savefig(out_path, dpi=150)
    plt.close(fig)
    print(f"  Saved → {out_path}")


# ── Plot 3: Confusion Matrices ────────────────────────────────────────────────

def plot_confusion_matrices(cms: dict, out_path: str):
    n    = len(cms)
    fig  = plt.figure(figsize=(5 * n, 5))
    gs   = gridspec.GridSpec(1, n, figure=fig, wspace=0.4)

    for i, (name, cm) in enumerate(cms.items()):
        ax = fig.add_subplot(gs[i])
        cm_arr = np.array(cm)
        sns.heatmap(cm_arr, annot=True, fmt="d", cmap="YlOrRd", ax=ax,
                    cbar=False, linewidths=0.5,
                    xticklabels=["Benign", "Malicious"],
                    yticklabels=["Benign", "Malicious"])
        ax.set_title(name, fontsize=12, pad=8)
        ax.set_xlabel("Predicted", fontsize=10)
        ax.set_ylabel("Actual", fontsize=10)

    fig.suptitle("Confusion Matrices at Optimal F1 Threshold", fontsize=14, y=1.02)
    fig.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  Saved → {out_path}")


# ── Entry point ───────────────────────────────────────────────────────────────

def evaluation_stage(anomalies_path: str = ANOMALIES_PARQUET,
                     figures_dir: str    = FIGURES_DIR,
                     metrics_path: str   = METRICS_JSON) -> dict:
    """
    End-to-end Stage 5 entry point.
    Requires: anomalies.parquet with ground_truth, if_anomaly_score,
              ocsvm_anomaly_score, ae_anomaly_score columns.
    """
    os.makedirs(figures_dir, exist_ok=True)

    print("Loading anomalies parquet …")
    df = pd.read_parquet(anomalies_path)
    print(f"    Shape: {df.shape}")

    if "ground_truth" not in df.columns:
        raise ValueError(
            "ground_truth column not found. "
            "Run stage0_label.py first, then re-run stage2_anomaly.py."
        )

    y_true = df["ground_truth"].values
    print(f"    Positives in anomaly set: {y_true.sum():,} / {len(y_true):,}")

    # Build score dict: {model_name: (score_array, config)}
    model_scores = {}
    for name, cfg in MODELS.items():
        if cfg["score_col"] not in df.columns:
            print(f"  Skipping {name} — column '{cfg['score_col']}' not found.")
            continue
        raw   = df[cfg["score_col"]].fillna(0).values
        score = -raw if cfg["negate"] else raw
        model_scores[name] = (score, cfg)

    if not model_scores:
        raise ValueError("No model score columns found in anomalies.parquet.")

    # ── Compute all metrics ───────────────────────────────────────────────────
    print("\n── Evaluation Metrics ─────────────────────────────────────")
    report = {}
    cms    = {}

    for name, (score, cfg) in model_scores.items():
        fpr_curve, tpr_curve, _ = roc_curve(y_true, score)
        roc_auc = auc(fpr_curve, tpr_curve)
        pr_auc  = average_precision_score(y_true, score)
        opt     = _optimal_threshold_metrics(y_true, score)
        cms[name] = opt["cm"]

        report[name] = {
            "roc_auc"   : round(roc_auc, 4),
            "pr_auc"    : round(pr_auc, 4),
            "f1"        : round(opt["f1"], 4),
            "precision" : round(opt["precision"], 4),
            "recall"    : round(opt["recall"], 4),
            "fpr"       : round(opt["fpr"], 4),
        }

        print(f"\n  {name}")
        print(f"    ROC-AUC   : {roc_auc:.4f}")
        print(f"    PR-AUC    : {pr_auc:.4f}")
        print(f"    F1 (opt)  : {opt['f1']:.4f}")
        print(f"    Precision : {opt['precision']:.4f}")
        print(f"    Recall    : {opt['recall']:.4f}")
        print(f"    FPR (opt) : {opt['fpr']:.4f}")

    print("────────────────────────────────────────────────────────────")

    # ── Generate figures ──────────────────────────────────────────────────────
    print("\n── Generating figures ─────────────────────────────────────")
    plot_roc_curves(y_true, model_scores,
                    os.path.join(figures_dir, "roc_curve_comparison.png"))
    plot_pr_curves(y_true, model_scores,
                   os.path.join(figures_dir, "pr_curve_comparison.png"))
    plot_confusion_matrices(cms,
                            os.path.join(figures_dir, "confusion_matrices.png"))

    # ── Save JSON report ──────────────────────────────────────────────────────
    with open(metrics_path, "w", encoding="utf-8") as f:
        json.dump(report, f, indent=2)
    print(f"\n  Metrics saved → {metrics_path}")

    return report

In [ ]:
metrics_report = evaluation_stage(
    anomalies_path = ANOMALIES_PARQUET,
    figures_dir    = FIGURES_DIR,
    metrics_path   = METRICS_JSON,
)


In [ ]:
# Display metrics comparison table
import json, pandas as pd
from IPython.display import display, Image

with open(METRICS_JSON) as f:
    report = json.load(f)

metrics_df = pd.DataFrame(report).T
metrics_df.index.name = "Model"
print("\n── Evaluation Metrics Summary ──────────────────────────")
display(metrics_df.style.background_gradient(cmap='RdYlGn', axis=0)
        .format("{:.4f}"))

# Show figures inline
print("\n── ROC Curves ──")
display(Image(FIGURES_DIR + "/roc_curve_comparison.png"))
print("\n── Precision-Recall Curves ──")
display(Image(FIGURES_DIR + "/pr_curve_comparison.png"))
print("\n── Confusion Matrices ──")
display(Image(FIGURES_DIR + "/confusion_matrices.png"))


---
## Stage 6 — LLM & RAG Evaluation (RAGAS)

Evaluates the quality of the LLM threat analyses using the
**RAGAS (Retrieval-Augmented Generation Assessment)** framework.

Two conditions are compared:
- **With RAG**: LLM answers generated using ChromaDB-retrieved context (MITRE ATT&CK, Sigma rules, etc.)
- **Without RAG** (baseline): Same questions, same LLM, but **no context** provided

| Metric | Interpretation |
|---|---|
| **Faithfulness** | Are claims grounded in retrieved docs? < 0.7 = hallucination |
| **Answer Relevancy** | Is the response on-topic for the anomaly? |
| **Context Precision** | Are the most useful RAG docs ranked highest? |
| **Context Recall** | Did the retriever surface all necessary information? |

> The Δ (delta) between with-RAG and without-RAG Faithfulness is the thesis's
> key quantitative contribution on the LLM side.


In [ ]:
"""
Stage 6: LLM & RAG Evaluation — RAGAS Framework
Evaluates the quality of the LLM threat analysis produced by Stage 4b
using the RAGAS (Retrieval-Augmented Generation Assessment) framework.

Four metrics are computed per analysis entry:
  • Faithfulness      — Are all claims grounded in the retrieved context?
                        Score < 0.7 is considered hallucinatory.
  • Answer Relevancy  — Is the response relevant to the anomaly question?
  • Context Precision — Are the most useful RAG docs ranked highest?
  • Context Recall    — Did the retriever surface all necessary information?

Key research contribution:
  A "no-RAG baseline" is also evaluated (same LLM, same questions, but
  retrieved_docs replaced with empty string). Comparing Faithfulness
  WITH RAG vs WITHOUT RAG quantitatively proves that the RAG pipeline
  reduces hallucinations — a compelling thesis finding.

Outputs:
  ├── llm_metrics.json               — Per-entry and aggregated RAGAS scores
  └── figures/ragas_radar_chart.png  — Radar chart of 4 metrics (with/without RAG)
  └── figures/faithfulness_comparison.png — Bar chart: RAG vs no-RAG Faithfulness
"""
import os
import json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from openai import OpenAI
from datasets import Dataset
from ragas import evaluate, RunConfig
from ragas.llms import llm_factory

# RAGAS v0.2+: metrics are classes that must be instantiated with an LLM.
# They are imported here but instantiated dynamically inside _run_ragas.
METRIC_COLS = ["faithfulness", "answer_relevancy", "context_precision", "context_recall"]

RESULTS_JSON    = "/content/data/llm_results.json"
LLM_METRICS_JSON = "/content/data/llm_metrics.json"
FIGURES_DIR     = "/content/data/figures"
OLLAMA_MODEL    = "llama3"
OLLAMA_BASE_URL = "http://localhost:11434"

plt.style.use("dark_background")


# ── Helpers ───────────────────────────────────────────────────────────────────

def _configure_ragas_llm(model: str = OLLAMA_MODEL):
    """
    Configure RAGAS LLM and embeddings for local Ollama (RAGAS v0.2+ API).

    LLM : llm_factory pointed at Ollama's OpenAI-compatible /v1 endpoint.
          api_key can be any non-empty string; Ollama ignores it.
    Emb : Langchain's HuggingFaceEmbeddings wrapper around all-MiniLM-L6-v2.
          sentence-transformers is already installed from Stage 3.
          This avoids routing embedding calls through Ollama entirely.
    """
    from langchain_huggingface import HuggingFaceEmbeddings as LangchainHFEmbeddings

    ollama_client = OpenAI(
        base_url=f"{OLLAMA_BASE_URL}/v1",
        api_key="ollama",
    )
    llm = llm_factory(model=model, client=ollama_client)
    emb = LangchainHFEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

    return llm, emb


def _build_ragas_dataset(results: list[dict],
                         no_rag: bool = False) -> Dataset:
    """
    Convert llm_results.json entries into a HuggingFace Dataset
    in the format expected by RAGAS.

    RAGAS expects columns:
        question   : str          — the question/prompt
        answer     : str          — the LLM's generated answer
        contexts   : list[str]    — the retrieved passages used
        ground_truth: str         — reference answer (optional for some metrics)
    """
    rows = []
    for r in results:
        question = (
            f"Analyse this security anomaly: "
            f"EventID {r.get('event_id')} on host {r.get('hostname')}. "
            f"Topic keywords: {r.get('topic_keywords', '')}."
        )
        answer   = (
            f"Threat Analysis: {r.get('threat_analysis', '')}\n"
            f"MITRE Technique: {r.get('mitre_technique', '')}\n"
            f"Remediation: {r.get('remediation_steps', '')}\n"
            f"Severity: {r.get('severity_rating', '')}"
        )
        # For the no-RAG baseline, replace context with empty string
        context_text = r.get("retrieved_docs", "") if not no_rag else ""
        contexts = [context_text] if context_text else ["No context provided."]

        # Ground truth: use MITRE technique as a minimal reference
        ground_truth = r.get("mitre_technique", "Unknown technique")

        rows.append({
            "question"    : question,
            "answer"      : answer,
            "contexts"    : contexts,
            "ground_truth": ground_truth,
        })
    return Dataset.from_list(rows)


def _run_ragas(dataset: Dataset, llm, emb) -> dict:
    """
    Run RAGAS evaluation sequentially (max_workers=1) to avoid overwhelming
    a local Ollama instance with concurrent requests, which causes TimeoutErrors.
    Returns a dict of averaged metric scores.
    """
    run_cfg = RunConfig(
        max_workers=1,    # sequential — critical for local Ollama
        timeout=90,       # 90 s per call — fail fast instead of stalling forever
        max_retries=1,    # no retry spiral; if it fails once, move on
    )
    # Instantiate metrics with the LLM (required by RAGAS v0.2).
    # n_adaptations=1 is critical for local Ollama: the default of 3 means
    # RAGAS requests 3 parallel generations per call, which Ollama cannot
    # satisfy, causing a silent retry-spiral that stalls the progress bar.
    from ragas.metrics.collections import (
        Faithfulness, AnswerRelevancy, ContextPrecision, ContextRecall
    )
    metrics = [
        Faithfulness(llm=llm, n_adaptations=1),
        AnswerRelevancy(llm=llm, embeddings=emb, n_adaptations=1),
        ContextPrecision(llm=llm, n_adaptations=1),
        ContextRecall(llm=llm, n_adaptations=1),
    ]

    result = evaluate(
        dataset,
        metrics=metrics,
        llm=llm,
        embeddings=emb,
        run_config=run_cfg,
        raise_exceptions=False,
    )

    # RAGAS returns an EvaluationResult — use to_pandas() to get per-row scores
    df   = result.to_pandas()
    cols = [c for c in METRIC_COLS if c in df.columns]
    return {k: round(float(v), 4) if pd.notna(v) else 0.0
            for k, v in df[cols].mean(numeric_only=True).items()}


# ── Radar chart ───────────────────────────────────────────────────────────────

def plot_radar_chart(with_rag: dict, without_rag: dict, out_path: str):
    metrics = ["faithfulness", "answer_relevancy", "context_precision", "context_recall"]
    labels  = ["Faithfulness", "Answer\nRelevancy", "Context\nPrecision", "Context\nRecall"]
    N       = len(metrics)
    angles  = [n / float(N) * 2 * np.pi for n in range(N)]
    angles += angles[:1]

    vals_with    = [with_rag.get(m, 0) for m in metrics]    + [with_rag.get(metrics[0], 0)]
    vals_without = [without_rag.get(m, 0) for m in metrics] + [without_rag.get(metrics[0], 0)]

    fig, ax = plt.subplots(1, 1, figsize=(7, 7),
                           subplot_kw=dict(polar=True))
    ax.set_facecolor("#0d1117")
    fig.patch.set_facecolor("#0d1117")

    ax.plot(angles, vals_with,    "o-", linewidth=2.5, color="#00b4d8", label="With RAG")
    ax.fill(angles, vals_with,    alpha=0.2, color="#00b4d8")
    ax.plot(angles, vals_without, "s--", linewidth=2.5, color="#e94560", label="Without RAG")
    ax.fill(angles, vals_without, alpha=0.15, color="#e94560")

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(labels, fontsize=12, color="white")
    ax.set_ylim(0, 1)
    ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
    ax.set_yticklabels(["0.2", "0.4", "0.6", "0.8", "1.0"], color="grey", fontsize=9)
    ax.grid(color="grey", alpha=0.3)
    ax.set_title("RAGAS Evaluation — RAG vs No-RAG", fontsize=14, color="white", pad=20)
    ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1), fontsize=11)

    fig.tight_layout()
    fig.savefig(out_path, dpi=150, facecolor=fig.get_facecolor())
    plt.close(fig)
    print(f"  Saved → {out_path}")


def plot_faithfulness_bar(with_rag: dict, without_rag: dict, out_path: str):
    metrics = ["faithfulness", "answer_relevancy", "context_precision", "context_recall"]
    labels  = ["Faithfulness", "Answer Relevancy", "Context Precision", "Context Recall"]
    x       = np.arange(len(metrics))
    w       = 0.35

    fig, ax = plt.subplots(figsize=(9, 5))
    fig.patch.set_facecolor("#0d1117")
    ax.set_facecolor("#0d1117")

    bars1 = ax.bar(x - w/2, [with_rag.get(m, 0) for m in metrics],
                   w, label="With RAG", color="#00b4d8", alpha=0.9)
    bars2 = ax.bar(x + w/2, [without_rag.get(m, 0) for m in metrics],
                   w, label="Without RAG", color="#e94560", alpha=0.9)

    ax.set_xlabel("RAGAS Metric", fontsize=12, color="white")
    ax.set_ylabel("Score (0–1)", fontsize=12, color="white")
    ax.set_title("RAGAS Scores: With RAG Context vs Without RAG Context",
                 fontsize=13, color="white", pad=12)
    ax.set_xticks(x)
    ax.set_xticklabels(labels, fontsize=10, color="white")
    ax.set_ylim(0, 1.1)
    ax.tick_params(colors="white")
    ax.legend(fontsize=11)
    ax.grid(axis="y", alpha=0.2)

    for bar in bars1:
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
                f"{bar.get_height():.2f}", ha="center", va="bottom",
                fontsize=9, color="white")
    for bar in bars2:
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
                f"{bar.get_height():.2f}", ha="center", va="bottom",
                fontsize=9, color="white")

    fig.tight_layout()
    fig.savefig(out_path, dpi=150, facecolor=fig.get_facecolor())
    plt.close(fig)
    print(f"  Saved → {out_path}")


# ── Entry point ───────────────────────────────────────────────────────────────

def llm_eval_stage(results_path: str   = RESULTS_JSON,
                   metrics_path: str   = LLM_METRICS_JSON,
                   figures_dir: str    = FIGURES_DIR,
                   model: str          = OLLAMA_MODEL,
                   max_samples: int    = 10) -> dict:  # 10 = ~80 Ollama calls; ~60–90 min on GPU
    """
    End-to-end Stage 6 entry point.
    Evaluates LLM analysis quality with and without RAG context using RAGAS.

    Args:
        max_samples: Cap on entries evaluated. RAGAS makes multiple LLM calls
                     per row (4 metrics × N rows), so keep this small for local
                     Ollama. 10 samples = ~80 LLM calls, runtime ~15-30 min.
                     Increase to 30 for the final thesis run.
    """
    os.makedirs(figures_dir, exist_ok=True)

    print(f"Loading LLM results from {results_path} …")
    with open(results_path, "r", encoding="utf-8") as f:
        results = json.load(f)

    # Subsample for evaluation speed
    if len(results) > max_samples:
        rng     = np.random.default_rng(42)
        indices = rng.choice(len(results), max_samples, replace=False)
        results = [results[i] for i in sorted(indices)]
    print(f"  Evaluating {len(results)} entries …")

    llm, emb = _configure_ragas_llm(model)

    # ── WITH RAG ─────────────────────────────────────────────────────────────
    print("\n── Evaluating WITH RAG context ────────────────────────────")
    ds_with  = _build_ragas_dataset(results, no_rag=False)
    scores_with = _run_ragas(ds_with, llm, emb)
    print("  Scores:", {k: f"{v:.4f}" for k, v in scores_with.items()})

    # ── WITHOUT RAG baseline ──────────────────────────────────────────────────
    print("\n── Evaluating WITHOUT RAG (baseline) ──────────────────────")
    ds_without  = _build_ragas_dataset(results, no_rag=True)
    scores_without = _run_ragas(ds_without, llm, emb)
    print("  Scores:", {k: f"{v:.4f}" for k, v in scores_without.items()})

    # ── Delta (RAG improvement) ───────────────────────────────────────────────
    delta = {k: round(scores_with.get(k, 0) - scores_without.get(k, 0), 4)
             for k in scores_with}
    print("\n── RAG Improvement (Δ) ─────────────────────────────────────")
    for k, v in delta.items():
        direction = "▲" if v >= 0 else "▼"
        print(f"  {k:25s}: {direction} {abs(v):.4f}")
    print("────────────────────────────────────────────────────────────")

    # ── Figures ───────────────────────────────────────────────────────────────
    print("\n── Generating RAGAS figures ───────────────────────────────")
    plot_radar_chart(
        scores_with, scores_without,
        os.path.join(figures_dir, "ragas_radar_chart.png")
    )
    plot_faithfulness_bar(
        scores_with, scores_without,
        os.path.join(figures_dir, "faithfulness_comparison.png")
    )

    # ── Save JSON ─────────────────────────────────────────────────────────────
    output = {
        "with_rag"   : scores_with,
        "without_rag": scores_without,
        "delta"      : delta,
        "n_samples"  : len(results),
    }
    with open(metrics_path, "w", encoding="utf-8") as f:
        json.dump(output, f, indent=2)
    print(f"\n  LLM metrics saved → {metrics_path}")

    return output

In [ ]:
llm_metrics = llm_eval_stage(
    results_path = RESULTS_JSON,
    metrics_path = LLM_METRICS_JSON,
    figures_dir  = FIGURES_DIR,
    model        = OLLAMA_MODEL,
    max_samples  = 10,       # 10 samples = ~80 Ollama calls; ~60–90 min on GPU
)


In [ ]:
import json
from IPython.display import display, Image

with open(LLM_METRICS_JSON) as f:
    llm_m = json.load(f)

print("── RAGAS Results ───────────────────────────────────────")
for condition in ["with_rag", "without_rag", "delta"]:
    print(f"\n  [{condition.upper()}]")
    for k, v in llm_m[condition].items():
        print(f"    {k:25s}: {v:.4f}")
print("────────────────────────────────────────────────────────")

display(Image(FIGURES_DIR + "/ragas_radar_chart.png"))
display(Image(FIGURES_DIR + "/faithfulness_comparison.png"))


---
## Results Dashboard

Summary visualisations of the LLM analysis output.


In [ ]:
import json, pandas as pd
import plotly.express as px
import plotly.graph_objects as go

with open(RESULTS_JSON) as f:
    results = json.load(f)

res_df = pd.DataFrame(results)
res_df = res_df.rename(columns={"mitre_technique": "Attack Technique", "remediation_steps": "Fixes"})
display(res_df[["topic_id","event_id","hostname","Attack Technique","Fixes","severity_rating"]].head(20))


In [ ]:
# Severity breakdown
sev_counts = res_df["severity_rating"].str.extract(r"(Critical|High|Medium|Low)")[0].value_counts()
fig = px.pie(
    values=sev_counts.values,
    names=sev_counts.index,
    title="Severity Distribution of Detected Anomalies",
    color=sev_counts.index,
    color_discrete_map={"Critical":"#e94560","High":"#f5a623","Medium":"#f0e130","Low":"#1db954"},
    template="plotly_dark",
    hole=0.4,
)
fig.update_layout(height=420)
fig.show()


In [ ]:
# MITRE technique frequency
tech_counts = res_df["Attack Technique"].value_counts().head(12)
fig = px.bar(
    x=tech_counts.values,
    y=tech_counts.index,
    orientation="h",
    title="Most Frequent MITRE ATT&CK Techniques",
    template="plotly_dark",
    color=tech_counts.values,
    color_continuous_scale="reds",
    height=500,
    labels={"x":"Count","y":"Technique"},
)
fig.update_layout(showlegend=False, yaxis=dict(autorange="reversed"))
fig.show()


In [ ]:
# Per-topic severity heatmap
import numpy as np

pivot = res_df.assign(
    sev_num=res_df["severity_rating"].str.extract(r"(Critical|High|Medium|Low)")[0].map(
        {"Critical":4,"High":3,"Medium":2,"Low":1}
    )
).groupby(["topic_id","event_id"])["sev_num"].mean().unstack(fill_value=0)

fig = go.Figure(go.Heatmap(
    z=pivot.values,
    x=[str(c) for c in pivot.columns],
    y=[f"Topic {r}" for r in pivot.index],
    colorscale="Reds",
    colorbar_title="Severity",
))
fig.update_layout(
    title="Severity Heatmap — Topic × EventID",
    template="plotly_dark",
    height=max(300, len(pivot)*40),
)
fig.show()


---
## Pipeline Complete

All outputs saved to `/content/data/`:
| File | Description |
|---|---|
| `normalized.parquet` | All ~1M events, flat schema |
| `labeled.parquet` | All events with `ground_truth` column (Stage 0) |
| `anomalies.parquet` | Anomalies with IF / OCSVM / AE scores (Stage 2) |
| `anomalies_with_topics.parquet` | Anomalies + BERTopic labels (Stage 3) |
| `chroma_db/` | Persistent vector store — 5 KB collections (Stage 4a) |
| `bertopic_model/` | Saved BERTopic model (Stage 3) |
| `llm_results.json` | Structured LLM threat analysis per anomaly (Stage 4b) |
| `metrics_report.json` | ROC-AUC / PR-AUC / F1 per model (Stage 5) |
| `llm_metrics.json` | RAGAS scores with/without RAG (Stage 6) |
| `figures/` | All thesis-ready PNG plots |
